In [1]:
import numpy as np
import pandas as pd
pd.set_option('future.no_silent_downcasting', True)

Clean up and spliting fact/dim aqi.csv

In [2]:
df_aqi = pd.read_csv('dataset/aqi.csv')

In [3]:
df_aqi

,date,state,area,number_of_monitoring_stations,prominent_pollutants,aqi_value,air_quality_status,unit,note
0,30-04-2025,Maharashtra,Amravati,2,PM10,78,Satisfactory,number_of_monitoring_stations in Absolute Numb...,NaN
1,30-04-2025,Bihar,Purnia,1,CO,56,Satisfactory,number_of_monitoring_stations in Absolute Numb...,NaN
2,30-04-2025,Madhya Pradesh,Katni,1,O3,98,Satisfactory,number_of_monitoring_stations in Absolute Numb...,NaN
3,30-04-2025,Chhattisgarh,Tumidih,1,PM10,103,Moderate,number_of_monitoring_stations in Absolute Numb...,NaN
4,30-04-2025,Assam,Byrnihat,1,PM2.5,61,Satisfactory,number_of_monitoring_stations in Absolute Numb...,NaN
...,...,...,...,...,...,...,...,...,...
235780,01-04-2022,Bihar,Arrah,1,PM10,210,Poor,number_of_monitoring_stations in Absolute Numb...,NaN
235781,01-04-2022,Rajasthan,Jaipur,3,"PM2.5,PM10",158,Moderate,number_of_monitoring_stations in Absolute Numb...,NaN
235782,01-04-2022,Maharashtra,Chandrapur,2,PM2.5,201,Poor,number_of_monitoring_stations in Absolute Numb...,NaN
235783,01-04-2022,Uttar Pradesh,Varanasi,4,PM10,128,Moderate,number_of_monitoring_stations in Absolute Numb...,NaN


drop unit and note column since these are not important

In [4]:
df_aqi.drop('note',axis=1,inplace=True)
df_aqi.drop('unit',axis=1,inplace=True)

convert data to right form date format and get date_id with corresponding date

In [5]:
df_aqi['date'] = pd.to_datetime(df_aqi['date'],dayfirst=True, errors='coerce')
df_aqi_clean = df_aqi.dropna(subset=['date']).copy()
df_aqi_clean['aqi_id'] = df_aqi_clean.reset_index().index + 1
df_aqi_clean

,date,state,area,number_of_monitoring_stations,prominent_pollutants,aqi_value,air_quality_status,aqi_id
0,2025-04-30,Maharashtra,Amravati,2,PM10,78,Satisfactory,1
1,2025-04-30,Bihar,Purnia,1,CO,56,Satisfactory,2
2,2025-04-30,Madhya Pradesh,Katni,1,O3,98,Satisfactory,3
3,2025-04-30,Chhattisgarh,Tumidih,1,PM10,103,Moderate,4
4,2025-04-30,Assam,Byrnihat,1,PM2.5,61,Satisfactory,5
...,...,...,...,...,...,...,...,...
235780,2022-04-01,Bihar,Arrah,1,PM10,210,Poor,235781
235781,2022-04-01,Rajasthan,Jaipur,3,"PM2.5,PM10",158,Moderate,235782
235782,2022-04-01,Maharashtra,Chandrapur,2,PM2.5,201,Poor,235783
235783,2022-04-01,Uttar Pradesh,Varanasi,4,PM10,128,Moderate,235784


get dim_Area with corresponding state

In [6]:
state_names = df_aqi_clean['state'].unique()
dict_state = {state_names[i]: i+1  for i in range(len(state_names))}

##3 value is in idsp state but not in aqi so adding 3 state more which is: Goa, Dadra and Nagar Haveli and Daman and Diu, Ladakh
idsp_state = {'Goa':33,'Dadra and Nagar Haveli and Daman and Diu': 34, 'Ladakh':35, 'Lakshadweep':36, 'All India':37}
dict_state.update(idsp_state)
#df_state = pd.DataFrame(dict_state.items(),columns=['state', 'state_id'])
#df_state.to_csv('processed_dataset/dim_state.csv', index=False,header=True)

In [7]:
dict_state

{'Maharashtra': 1,
 'Bihar': 2,
 'Madhya Pradesh': 3,
 'Chhattisgarh': 4,
 'Assam': 5,
 'Manipur': 6,
 'Kerala': 7,
 'West Bengal': 8,
 'Odisha': 9,
 'Karnataka': 10,
 'Gujarat': 11,
 'Uttarakhand': 12,
 'Tamil Nadu': 13,
 'Andhra Pradesh': 14,
 'Rajasthan': 15,
 'Uttar Pradesh': 16,
 'Punjab': 17,
 'Mizoram': 18,
 'Chandigarh': 19,
 'Telangana': 20,
 'Puducherry': 21,
 'Meghalaya': 22,
 'Himachal Pradesh': 23,
 'Jharkhand': 24,
 'Haryana': 25,
 'Arunachal Pradesh': 26,
 'Nagaland': 27,
 'Tripura': 28,
 'Delhi': 29,
 'Andaman and Nicobar Islands': 30,
 'Sikkim': 31,
 'Jammu and Kashmir': 32,
 'Goa': 33,
 'Dadra and Nagar Haveli and Daman and Diu': 34,
 'Ladakh': 35,
 'Lakshadweep': 36,
 'All India': 37}

In [8]:
#get the pair area - state to look up district in gemini
'''
temp = df_aqi[['area','state']]
temp_test = temp.drop_duplicates()
temp_test.to_csv('togetdistrict.csv',index=False,header=True)
'''
#using google gemini to find corresponding district for each area in state. the file "area_state_district.csv" was pulled from google gemini
#this cell is using to cleaned the district text received from google gemini

"\ntemp = df_aqi[['area','state']]\ntemp_test = temp.drop_duplicates()\ntemp_test.to_csv('togetdistrict.csv',index=False,header=True)\n"

In [9]:
area = df_aqi_clean['area'].unique()
dict_area = {area[i]: i+1 for i in range(len(area))}
dict_area
#unique_area_state.to_csv('processed_dataset/dim_area.csv', index=True, header=True)

{'Amravati': 1,
 'Purnia': 2,
 'Katni': 3,
 'Tumidih': 4,
 'Byrnihat': 5,
 'Imphal': 6,
 'Kollam': 7,
 'Barrackpore': 8,
 'Nayagarh': 9,
 'Nalbari': 10,
 'Hubballi': 11,
 'Ahmedabad': 12,
 'Dehradun': 13,
 'Vellore': 14,
 'Ulhasnagar': 15,
 'Chengalpattu': 16,
 'Tirupati': 17,
 'Dindigul': 18,
 'Kadapa': 19,
 'Thane': 20,
 'Rishikesh': 21,
 'Raipur': 22,
 'Ranipet': 23,
 'Samastipur': 24,
 'Jabalpur': 25,
 'Kishanganj': 26,
 'Saharsa': 27,
 'Yadgir': 28,
 'Dungarpur': 29,
 'Meerut': 30,
 'Sawai Madhopur': 31,
 'Ghaziabad': 32,
 'Pithampur': 33,
 'Kanpur': 34,
 'Chennai': 35,
 'Nagaur': 36,
 'Chittoor': 37,
 'Kalyan': 38,
 'Mira Bhayandar': 39,
 'Bhubaneswar': 40,
 'Churu': 41,
 'Jodhpur': 42,
 'Katihar': 43,
 'Pali': 44,
 'Balasore': 45,
 'Sikar': 46,
 'Jalandhar': 47,
 'Tirumala': 48,
 'Angul': 49,
 'Patiala': 50,
 'Pimpri Chinchwad': 51,
 'Vatva': 52,
 'Chamarajanagar': 53,
 'Jaisalmer': 54,
 'Keonjhar': 55,
 'Chittorgarh': 56,
 'Chikkaballapur': 57,
 'Bhilwara': 58,
 'Arrah': 59,
 '

In [10]:
df_aqi_clean.loc[:,'area'] = df_aqi_clean.loc[:,'area'].replace(dict_area)
df_aqi_clean.loc[:,'state'] = df_aqi_clean.loc[:,'state'].replace(dict_state)

In [11]:
bridge_row = []
for index, row in df_aqi_clean.iterrows():
    pollutants = row['prominent_pollutants'].split(',')
    for i in pollutants:
        bridge_row.append({'aqi_id': row['aqi_id'], 'pollutant_code': i})
df_aqi_clean.drop('prominent_pollutants',axis=1,inplace=True)

In [12]:
df_aqi_clean.drop('air_quality_status', axis=1, inplace=True)

In [13]:
df_la = pd.read_csv('processed_dataset/bridge_linking_area.csv')
df_la

,area_id,state_id,district_id,asd_id
0,1,1,1,1
1,2,2,2,2
2,3,3,3,3
3,4,4,4,4
4,5,5,5,5
...,...,...,...,...
287,287,7,175,288
288,288,32,252,289
289,289,7,253,290
290,290,2,254,291


In [14]:
df_aqi_clean

,date,state,area,number_of_monitoring_stations,aqi_value,aqi_id
0,2025-04-30,1,1,2,78,1
1,2025-04-30,2,2,1,56,2
2,2025-04-30,3,3,1,98,3
3,2025-04-30,4,4,1,103,4
4,2025-04-30,5,5,1,61,5
...,...,...,...,...,...,...
235780,2022-04-01,2,59,1,210,235781
235781,2022-04-01,15,166,3,158,235782
235782,2022-04-01,1,174,2,201,235783
235783,2022-04-01,16,204,4,128,235784


In [15]:
##bridge_pollutant_aqi = pd.DataFrame(bridge_row)
##bridge_pollutant_aqi.to_csv('processed_dataset/bride_pollutant_aqi.csv', index=False,header=True)

In [16]:
df_aqi_clean.rename(columns={'area':'area_id'},inplace=True)

In [17]:
df_aqi_clean.rename(columns={'state':'state_id'},inplace=True)

In [18]:
df_aqi_clean

,date,state_id,area_id,number_of_monitoring_stations,aqi_value,aqi_id
0,2025-04-30,1,1,2,78,1
1,2025-04-30,2,2,1,56,2
2,2025-04-30,3,3,1,98,3
3,2025-04-30,4,4,1,103,4
4,2025-04-30,5,5,1,61,5
...,...,...,...,...,...,...
235780,2022-04-01,2,59,1,210,235781
235781,2022-04-01,15,166,3,158,235782
235782,2022-04-01,1,174,2,201,235783
235783,2022-04-01,16,204,4,128,235784


In [19]:
def find_asd_id(s,a,df_la):
    tmp = df_la[(df_la['state_id'] == s) & (df_la['area_id'] == a)]
    return tmp['asd_id'].iloc[0]
asd = []
for index, row in df_aqi_clean.iterrows():
    asd.append(find_asd_id(row['state_id'],row['area_id'],df_la))

In [20]:
df_aqi_clean['asd_id'] = asd


In [21]:
df_aqi_clean.drop('state_id',axis=1,inplace=True)
df_aqi_clean.drop('area_id',axis=1,inplace=True)
df_aqi_clean


,date,number_of_monitoring_stations,aqi_value,aqi_id,asd_id
0,2025-04-30,2,78,1,1
1,2025-04-30,1,56,2,2
2,2025-04-30,1,98,3,3
3,2025-04-30,1,103,4,4
4,2025-04-30,1,61,5,5
...,...,...,...,...,...
235780,2022-04-01,1,210,235781,59
235781,2022-04-01,3,158,235782,167
235782,2022-04-01,2,201,235783,175
235783,2022-04-01,4,128,235784,205


In [22]:
#df_aqi_clean.to_csv('processed_dataset/fact_aqi.csv',index=False,header=True)

process idsp

In [10]:
df_idsp = pd.read_csv('dataset/idsp.csv')
df_idsp

,year,week,outbreak_starting_date,reporting_date,state,district,disease_illness_name,status,cases,deaths,unit,note
0,2025,16,15-04-2025,15-04-2025,Andhra Pradesh,Kakinada,Acute Diarrheal Disease,Reported,22,0,"cases in absolute number, deaths in absolute n...",NaN
1,2025,16,15-04-2025,17-04-2025,Assam,Biswanath,Chickenpox,Reported,1,1,"cases in absolute number, deaths in absolute n...",NaN
2,2025,16,19-04-2025,20-04-2025,Assam,Dhemaji,Food Poisoning,Reported,16,0,"cases in absolute number, deaths in absolute n...",NaN
3,2025,16,19-04-2025,19-04-2025,Bihar,Gopalganj,Fever with Rash,Reported,5,0,"cases in absolute number, deaths in absolute n...",NaN
4,2025,16,12/4/2025,15-04-2025,Bihar,Madhubani,Acute Diarrheal Disease,Reported,21,0,"cases in absolute number, deaths in absolute n...",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
6469,2022,13,31-03-2022,1/4/2022,Tamil Nadu,Krishnagiri,Food Poisoning,Reported in Same Week,18,0,"cases in absolute number, deaths in absolute n...",NaN
6470,2022,13,30-03-2022,30-03-2022,Tamil Nadu,Pudukkottai,Dengue,Reported in Same Week,15,0,"cases in absolute number, deaths in absolute n...",NaN
6471,2022,13,28-03-2022,4/4/2022,Tamil Nadu,Tiruchirappalli,Chickenpox,Reported in Same Week,24,0,"cases in absolute number, deaths in absolute n...",NaN
6472,2022,13,4/2/2022,4/3/2022,Telangana,Jagtial,Food Poisoning,Reported in Same Week,25,0,"cases in absolute number, deaths in absolute n...",NaN


drop unit, note and week (is not reliable)

In [12]:
df_idsp.drop('week',axis=1,inplace=True)
df_idsp.drop('unit',axis=1,inplace=True)
df_idsp.drop('note',axis=1,inplace=True)
df_idsp.drop('status',axis=1,inplace=True)
df_idsp.dropna(subset=['district'], inplace=True)
df_idsp.dropna(subset=['disease_illness_name'],inplace=True)
df_idsp

,year,outbreak_starting_date,reporting_date,state,district,disease_illness_name,cases,deaths
0,2025,15-04-2025,15-04-2025,Andhra Pradesh,Kakinada,Acute Diarrheal Disease,22,0
1,2025,15-04-2025,17-04-2025,Assam,Biswanath,Chickenpox,1,1
2,2025,19-04-2025,20-04-2025,Assam,Dhemaji,Food Poisoning,16,0
3,2025,19-04-2025,19-04-2025,Bihar,Gopalganj,Fever with Rash,5,0
4,2025,12/4/2025,15-04-2025,Bihar,Madhubani,Acute Diarrheal Disease,21,0
...,...,...,...,...,...,...,...,...
6469,2022,31-03-2022,1/4/2022,Tamil Nadu,Krishnagiri,Food Poisoning,18,0
6470,2022,30-03-2022,30-03-2022,Tamil Nadu,Pudukkottai,Dengue,15,0
6471,2022,28-03-2022,4/4/2022,Tamil Nadu,Tiruchirappalli,Chickenpox,24,0
6472,2022,4/2/2022,4/3/2022,Telangana,Jagtial,Food Poisoning,25,0


In [2]:
air_illness = ['Chickenpox', 'Measles', 'Mumps', 'HMPV', 'Mpox', 'Meningitis', 
'Pertussis', 'Rubella', 'Measles and Rubella', 'Diphtheria', 
'Hand Foot and Mouth Disease', 'Nipah Virus', 
'Poliomyelitis (Vaccine-Derived Polio Virus)', 
'ARI Influenza Like Illness(ILI)', 'Anthrax', 'Influenza', 
'Suspected Mumps', 'Melioidosis', 'Monkey Pox', 'Brucellosis', 
'Suspected Anthrax', 'Adenovirus', 'Acute Flaccid Paralysis', 
'Chickenpox and Measles', 'H3N2', 
'Fever and Upper Respiratandy Tract Infection (URTI)', 
'Allergic Conjunctivitis', 'Fever with Rash (Enterovirus)', 
'Chemical Gas Poisoning', 'Seasonal Influenza', 'Swine Flu (H1N1)', 
'Acute Respiratandy Illness', 'Influenza A']

In [26]:
df_idsp_clean = df_idsp[df_idsp['disease_illness_name'].isin(air_illness)]
df_idsp_clean.reset_index(drop=True,inplace=True)

In [27]:
df_idsp_clean.loc[:,'disease_illness_name'] = df_idsp_clean.loc[:,'disease_illness_name'].replace({'Monkey Pox': 'Mpox'})
df_idsp_clean.loc[:,'disease_illness_name'] = df_idsp_clean.loc[:,'disease_illness_name'].replace({'Suspected Anthrax': 'Anthrax'})
df_idsp_clean.loc[:,'disease_illness_name'] = df_idsp_clean.loc[:,'disease_illness_name'].replace({'Chickenpox and Measles': 'Chickenpox'})
df_idsp_clean.loc[:,'disease_illness_name'] = df_idsp_clean.loc[:,'disease_illness_name'].replace({'Suspected Mumps': 'Mumps'})
df_idsp_clean.loc[:,'disease_illness_name'] = df_idsp_clean.loc[:,'disease_illness_name'].replace({'Measles and Rubella': 'Rubella'})

In [3]:
disease_to_cluster_map = {
    # Cluster 1: Influenza and Its Subtypes
    'Influenza': 'Influenza and Its Subtypes',
    'H3N2': 'Influenza and Its Subtypes',
    'Seasonal Influenza': 'Influenza and Its Subtypes',
    'Swine Flu (H1N1)': 'Influenza and Its Subtypes',
    'Influenza A': 'Influenza and Its Subtypes',

    # Cluster 2: General Acute Respiratory Infections
    'ARI Influenza Like Illness(ILI)': 'General Acute Respiratory Infections',
    'Fever and Upper Respiratandy Tract Infection (URTI)': 'General Acute Respiratory Infections',
    'Acute Respiratandy Illness': 'General Acute Respiratory Infections',

    # Cluster 3: Common Airborne Viral Illnesses
    'Chickenpox': 'Common Airborne Viral Illnesses',
    'Measles': 'Common Airborne Viral Illnesses',
    'Mumps': 'Common Airborne Viral Illnesses',
    'Rubella': 'Common Airborne Viral Illnesses',
    'Hand Foot and Mouth Disease': 'Common Airborne Viral Illnesses',
    'Fever with Rash (Enterovirus)': 'Common Airborne Viral Illnesses',
    'Mpox': 'Common Airborne Viral Illnesses',

    # Cluster 4: Severe Airborne Bacterial Infections
    'Pertussis': 'Severe Airborne Bacterial Infections',
    'Diphtheria': 'Severe Airborne Bacterial Infections',
    'Anthrax': 'Severe Airborne Bacterial Infections',
    'Melioidosis': 'Severe Airborne Bacterial Infections',
    'Brucellosis': 'Severe Airborne Bacterial Infections',

    # Cluster 5: Neurological Complications
    'Meningitis': 'Neurological Complications',
    'Poliomyelitis (Vaccine-Derived Polio Virus)': 'Neurological Complications',
    'Acute Flaccid Paralysis': 'Neurological Complications',

    # Cluster 6: Other Specific Respiratory Viruses
    'HMPV': 'Other Specific Respiratory Viruses',
    'Adenovirus': 'Other Specific Respiratory Viruses',
    'Nipah Virus': 'Other Specific Respiratory Viruses',

    # Cluster 7: Non-Infectious Air Quality Conditions
    'Allergic Conjunctivitis': 'Non-Infectious Air Quality Conditions',
    'Chemical Gas Poisoning': 'Non-Infectious Air Quality Conditions',
}

In [4]:
cluster_to_id_map = {
    'Influenza and Its Subtypes': 1,
    'General Acute Respiratory Infections': 2,
    'Common Airborne Viral Illnesses': 3,
    'Severe Airborne Bacterial Infections': 4,
    'Neurological Complications': 5,
    'Other Specific Respiratory Viruses': 6,
    'Non-Infectious Air Quality Conditions': 7,
}
ctid = pd.DataFrame(cluster_to_id_map,columns=["ilness","illness_id"])
ctid.to_csv("dim_illness.csv", index=False, header=True)

In [30]:
df_idsp_clean.loc[:,'disease_illness_name'] = df_idsp_clean.loc[:,'disease_illness_name'].map(disease_to_cluster_map)
df_idsp_clean.loc[:,'disease_illness_name'] = df_idsp_clean.loc[:,'disease_illness_name'].map(cluster_to_id_map)


In [31]:
df_idsp_clean

,outbreak_starting_date,reporting_date,state,district,disease_illness_name,cases,deaths
0,15-04-2025,17-04-2025,Assam,Biswanath,3,1,1
1,15-04-2025,16-04-2025,Bihar,Patna,3,8,1
2,19-04-2025,19-04-2025,Bihar,Vaishali,3,9,0
3,16-04-2025,16-04-2025,Gujarat,Narmada,3,17,0
4,17-04-2025,17-04-2025,Haryana,Nuh,3,28,0
...,...,...,...,...,...,...,...
1241,27-03-2022,NaN,Karnataka,Chikkamagaluru,3,17,0
1242,4/2/2022,18-04-2022,Meghalaya,South West Garo Hills,3,32,0
1243,29-03-2022,30-03-2022,Jharkhand,Godda,3,2,0
1244,16-12-2021,29-03-2022,Jharkhand,Sahebganj,3,63,0


In [32]:
df_idsp_clean.loc[:,'district'] = df_idsp_clean.loc[:,'district'].str.replace(r'[\r\n]+', '', regex=True)

In [33]:
df_idsp_clean.loc[:,'state'].replace(dict_state,inplace=True)

C:\Users\HP\AppData\Local\Temp\ipykernel_11764\3551054009.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_idsp_clean.loc[:,'state'].replace(dict_state,inplace=True)
C:\Users\HP\AppData\Local\Temp\ipykernel_11764\3551054009.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_idsp_clean.loc[:,'state'].replace(dict_state,inplace=True)


In [34]:
df_idsp_clean.loc[:,'district'] = df_idsp_clean.loc[:,'district'].replace('East Singhbhum', 'East Singhbum')
df_idsp_clean.loc[:,'district']= df_idsp_clean.loc[:,'district'].replace('Shri Muktsar Sahib', 'Sri Muktsar Sahib')
df_idsp_clean.loc[:,'district'] = df_idsp_clean.loc[:,'district'].replace('Purbi Champaran', 'Eastern Champaran')
df_idsp_clean.loc[:,'district']= df_idsp_clean.loc[:,'district'].replace('Pashchim Champaran', 'Western Champaran')
df_idsp_clean.loc[:,'district'] = df_idsp_clean.loc[:,'district'].replace('Dhumka', 'Dumka')
df_idsp_clean.loc[:,'district'] = df_idsp_clean.loc[:,'district'].replace('Jahanabad', 'Jehanabad')
df_idsp_clean.loc[:,'district'] = df_idsp_clean.loc[:,'district'].replace('Trivandrum', 'Thiruvananthapuram')
df_idsp_clean.loc[:,'district'] = df_idsp_clean.loc[:,'district'].replace('Leh Ladakh', 'Ladakh')
df_idsp_clean.loc[:,'district'] = df_idsp_clean.loc[:,'district'].replace('24 Paraganas North', 'North 24 Parganas')
df_idsp_clean.loc[:,'district'] = df_idsp_clean.loc[:,'district'].replace('Aizawl West', 'Aizawl')
df_idsp_clean.loc[:,'district'] = df_idsp_clean.loc[:,'district'].replace('West', 'West Delhi')



In [35]:
idsp_unique = df_idsp_clean['district'].unique()
idsp_unique

array(['Biswanath', 'Patna', 'Vaishali', 'Narmada', 'Nuh', 'Giridih',
       'Simdega', 'Datia', 'Morena', 'Seoni', 'East Khasi Hills',
       'Alipurduar', 'Dhanbad', 'Godda', 'Jamtara', 'Niwari', 'Ri Bhoi',
       'Ganjam', 'Erode', 'Madhubani', 'East Singhbum',
       'North Garo Hills', 'Kallakurichi', 'Sivaganga', 'Tiruchirappalli',
       'Koraput', 'Agra', 'Dumka', 'Khunti', 'Ranchi', 'Damoh',
       'Narsinghpur', 'Coimbatore', 'Dharmapuri', 'Dindigul', 'Theni',
       'Bilaspur', 'Shivpuri', 'Ukhrul', 'South West Garo Hills',
       'Bongaigaon', 'Jamnagar', 'Jammu', 'Ernakulam', 'Malappuram',
       'Salem', 'Howrah', 'Bhopal', 'Charaideo', 'Gwalior', 'Jalgaon',
       'Sri Muktsar Sahib', 'Mayiladuthurai', 'Tirupathur', 'Bagalkot',
       'Alappuzha', 'Kottayam', 'Lawngtlai', 'Jehanabad',
       'Pathanamthitta', 'Wayanad', 'Chhatarpur', 'Latur', 'Pudukkottai',
       'Ramanathapuram', 'Thanjavur', 'Vellore', 'Bhadohi', 'Konaseema',
       'Chirang', 'Narayanpur', 'Hazaribag

Create fact to track between area and district

In [36]:
df_ad = pd.read_csv('area_state_district.csv')
district = df_ad['district'].unique()
new_district = list(district) + [d for d in idsp_unique if d not in district]
dict_district = {new_district[i]: i+1 for i in range(len(new_district))}
df_ad.loc[:,'area'] = df_ad.loc[:,'area'].replace(dict_area)
df_ad.loc[:,'district'] = df_ad.loc[:,'district'].replace(dict_district)
df_ad.loc[:,'state'] = df_ad.loc[:,'state'].replace(dict_state)
df_ad.loc[:,'area'] = df_ad.loc[:,'area'].replace(dict_area)
df_ad.loc[:,'district'] = df_ad.loc[:,'district'].replace(dict_district)
df_ad.loc[:,'state'] = df_ad.loc[:,'state'].replace(dict_state)
df_ad.rename(columns={'area':'area_id','district':'district_id','state':'state_id'},inplace=True)
df_ad['asd_id'] = df_ad.reset_index().index + 1
#df_ad.to_csv('processed_dataset/bridge_linking_area.csv',index=False,header=True)

In [37]:
##normalize date
df_idsp_clean.loc[:,'outbreak_starting_date'] = df_idsp_clean.loc[:,'outbreak_starting_date'].astype(str).str.replace('/','-',regex=False)
df_idsp_clean.loc[:,'outbreak_starting_date'] = pd.to_datetime(df_idsp_clean.loc[:,'outbreak_starting_date'], format='%d-%m-%Y', errors='coerce').dt.date
df_idsp_clean.loc[:,'reporting_date'] = df_idsp_clean.loc[:,'reporting_date'].astype(str).str.replace('/','-',regex=False)
df_idsp_clean.loc[:,'reporting_date'] = pd.to_datetime(df_idsp_clean.loc[:,'reporting_date'],format='%d-%m-%Y', errors='coerce').dt.date

In [38]:
#encode district
df_idsp_clean.loc[:,'district'] = df_idsp_clean.loc[:,'district'].replace(dict_district)

In [39]:
df_idsp_clean['idsp_id'] = df_idsp_clean.reset_index().index + 1

C:\Users\HP\AppData\Local\Temp\ipykernel_11764\3332107047.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_idsp_clean['idsp_id'] = df_idsp_clean.reset_index().index + 1


In [40]:
#df_idsp_clean.to_csv('processed_dataset/fact_idsp.csv',index=False,header=True)


In [41]:
df_district = pd.DataFrame(dict_district.items(),columns=[['district','district_id']])
##df_district.to_csv('processed_dataset/dim_district.csv',index=False,header=True)

PROCESSING VAHAN

In [42]:
df_vahan = pd.read_csv('dataset/vahan.csv')
df_vahan

,year,month,state,rto,vehicle_class,fuel,value,unit,note
0,2025,April,Andaman and Nicobar Islands,All Vahan Running Office,BUS,DIESEL,2,value in Absolute Number,NaN
1,2025,April,Andaman and Nicobar Islands,All Vahan Running Office,GOODS CARRIER,DIESEL,23,value in Absolute Number,NaN
2,2025,April,Andaman and Nicobar Islands,All Vahan Running Office,GOODS CARRIER,PETROL,1,value in Absolute Number,NaN
3,2025,April,Andaman and Nicobar Islands,All Vahan Running Office,M-CYCLE/SCOOTER,ELECTRIC(BOV),1,value in Absolute Number,NaN
4,2025,April,Andaman and Nicobar Islands,All Vahan Running Office,M-CYCLE/SCOOTER,PETROL,387,value in Absolute Number,NaN
...,...,...,...,...,...,...,...,...,...
64836,2022,April,West Bengal,All Vahan Running Office,THREE WHEELER (PASSENGER),PETROL/CNG,3,value in Absolute Number,NaN
64837,2022,April,West Bengal,All Vahan Running Office,THREE WHEELER (PASSENGER),PETROL/LPG,82,value in Absolute Number,NaN
64838,2022,April,West Bengal,All Vahan Running Office,TRACTOR (COMMERCIAL),DIESEL,247,value in Absolute Number,NaN
64839,2022,April,West Bengal,All Vahan Running Office,TRAILER (COMMERCIAL),NOT APPLICABLE,60,value in Absolute Number,NaN


In [43]:
df_vahan.drop('rto',axis=1,inplace=True)
df_vahan.drop('unit',axis=1,inplace=True)
df_vahan.drop('note',axis=1,inplace=True)

In [44]:
df_vahan_clean = df_vahan.copy()

In [45]:
df_vahan_clean.loc[:,'state'] = df_vahan_clean['state'].replace(dict_state)
df_vahan_clean.rename(columns={'state':'state_id'}, inplace=True)

In [46]:
df_vahan_clean.loc[:,'fuel'] = df_vahan_clean['fuel'].replace('PURE EV','ELECTRIC(BOV)')
df_vahan_clean.loc[:,'fuel'] = df_vahan_clean['fuel'].replace('STRONG HYBRID EV','PETROL/HYBRID')

In [47]:
vehicle_fuel = df_vahan_clean[['vehicle_class','fuel']]
vehicle_fuel = vehicle_fuel.drop_duplicates()
vehicle_fuel.reset_index(drop=True,inplace=True)
#vehicle_fuel.to_csv('vehicles.csv',index=False)

In [48]:
vehicle_fuel

,vehicle_class,fuel
0,BUS,DIESEL
1,GOODS CARRIER,DIESEL
2,GOODS CARRIER,PETROL
3,M-CYCLE/SCOOTER,ELECTRIC(BOV)
4,M-CYCLE/SCOOTER,PETROL
...,...,...
328,EXCAVATOR (NT),CNG ONLY
329,RECOVERY VEHICLE,DUAL DIESEL/CNG
330,OMNI BUS (PRIVATE USE),ELECTRIC(BOV)
331,TRAILER (AGRICULTURAL),ELECTRIC(BOV)


In [49]:
vehicle_groups = {
    'Bus': [
        'BUS', 'OMNI BUS', 'OMNI BUS (PRIVATE USE)', 'SCHOOL BUS',
        'EDUCATIONAL INSTITUTION BUS', 'PRIVATE SERVICE VEHICLE',
        'PRIVATE SERVICE VEHICLE (INDIVIDUAL USE)'
    ],
    'Taxi & Cab': [
        'MOTOR CAB', 'MAXI CAB', 'LUXURY CAB'
    ],
    'Motorcycle & Scooter': [
        'M-CYCLE/SCOOTER', 'MOPED', 'MOTORISED CYCLE (CC > 25CC)',
        'M-CYCLE/SCOOTER-WITH SIDE CAR', 'MOTOR CYCLE/SCOOTER-SIDECAR(T)',
        'MOTOR CYCLE/SCOOTER-WITH TRAILER','MOTOR CYCLE/SCOOTER-USED FOR HIRE'
    ],
    'Three & Four-Wheeler': [
        'THREE WHEELER (GOODS)', 'THREE WHEELER (PASSENGER)',
        'THREE WHEELER (PERSONAL)', 'E-RICKSHAW WITH CART (G)', 'E-RICKSHAW(P)',
        'QUADRICYCLE (COMMERCIAL)', 'QUADRICYCLE (PRIVATE)'
    ],
    'Car & Recreational Vehicle': [
        'MOTOR CAR', 'CAMPER VAN / TRAILER', 'CAMPER VAN / TRAILER (PRIVATE USE)',
        'MOTOR CARAVAN','TRAILER FOR PERSONAL USE', 'AUXILIARY TRAILER'
    ],
    'Goods & Light Commercial Van': [
        'GOODS CARRIER', 'CASH VAN', 'BREAKDOWN VAN', 'MOBILE WORKSHOP',
        'MOBILE CANTEEN', 'LIBRARY VAN', 'VEHICLE FITTED WITH COMPRESSOR',
        'VEHICLE FITTED WITH GENERATOR'
    ],
    'Heavy & Articulated Vehicle': [
        'ARTICULATED VEHICLE', 'TRACTOR (COMMERCIAL)', 'PULLER TRACTOR',
        'SEMI-TRAILER (COMMERCIAL)', 'TRACTOR-TROLLEY(COMMERCIAL)', 'DUMPER','TRAILER (COMMERCIAL)'
    ],
    'Construction & Industrial Equipment': [
        'CONSTRUCTION EQUIPMENT VEHICLE', 'CONSTRUCTION EQUIPMENT VEHICLE (COMMERCIAL)',
        'EARTH MOVING EQUIPMENT', 'EXCAVATOR (NT)', 'EXCAVATOR (COMMERCIAL)',
        'BULLDOZER', 'ROAD ROLLER', 'CRANE MOUNTED VEHICLE', 'FORK LIFT',
        'VEHICLE FITTED WITH RIG', 'TOWER WAGON'
    ],
    'Agricultural Vehicle': [
        'AGRICULTURAL TRACTOR', 'HARVESTER', 'POWER TILLER',
        'TRAILER (AGRICULTURAL)'
    ],
    'Special & Emergency Vehicle': [
        'AMBULANCE', 'ANIMAL AMBULANCE', 'MOBILE CLINIC', 'FIRE TENDERS',
        'FIRE FIGHTING VEHICLE', 'SNORKED LADDERS', 'RECOVERY VEHICLE',
        'TOW TRUCK', 'HEARSES', 'TREE TRIMMING VEHICLE',
        'ARMOURED/SPECIALISED VEHICLE', 'ADAPTED VEHICLE'
    ]
}

In [50]:
df_vehicel_groups = pd.DataFrame(vehicle_groups.items(),columns=['class','vehicle'])

In [51]:
df_vehicel_groups_splited = df_vehicel_groups.explode('vehicle')
df_vehicel_groups_splited.reset_index(inplace=True,drop=True)
df_vehicel_groups_splited

,class,vehicle
0,Bus,BUS
1,Bus,OMNI BUS
2,Bus,OMNI BUS (PRIVATE USE)
3,Bus,SCHOOL BUS
4,Bus,EDUCATIONAL INSTITUTION BUS
...,...,...
67,Special & Emergency Vehicle,TOW TRUCK
68,Special & Emergency Vehicle,HEARSES
69,Special & Emergency Vehicle,TREE TRIMMING VEHICLE
70,Special & Emergency Vehicle,ARMOURED/SPECIALISED VEHICLE


In [52]:
merged_cvf = pd.merge(df_vehicel_groups_splited,vehicle_fuel,left_on='vehicle',right_on='vehicle_class')

In [67]:
merged_cvf.drop('vehicle_class',axis=1,inplace=True)
merged_cvf

,class,vehicle,fuel
0,Bus,BUS,DIESEL
1,Bus,BUS,CNG ONLY
2,Bus,BUS,ELECTRIC(BOV)
3,Bus,BUS,PETROL
4,Bus,BUS,FUEL CELL HYDROGEN
...,...,...,...
328,Special & Emergency Vehicle,ADAPTED VEHICLE,PETROL/CNG
329,Special & Emergency Vehicle,ADAPTED VEHICLE,ELECTRIC(BOV)
330,Special & Emergency Vehicle,ADAPTED VEHICLE,NOT APPLICABLE
331,Special & Emergency Vehicle,ADAPTED VEHICLE,DIESEL/HYBRID


In [56]:
mapping = {
    old_class: new_class
    for new_class,old_class_list in vehicle_groups.items()
    for old_class in old_class_list
}
df_vahan_clean.loc[:,'vehicle_class'] = df_vahan_clean['vehicle_class'].map(mapping).fillna('Others')

In [57]:
df_vahan_clean

,year,month,state_id,vehicle_class,fuel,value
0,2025,April,30,Bus,DIESEL,2
1,2025,April,30,Goods & Light Commercial Van,DIESEL,23
2,2025,April,30,Goods & Light Commercial Van,PETROL,1
3,2025,April,30,Motorcycle & Scooter,ELECTRIC(BOV),1
4,2025,April,30,Motorcycle & Scooter,PETROL,387
...,...,...,...,...,...,...
64836,2022,April,8,Three & Four-Wheeler,PETROL/CNG,3
64837,2022,April,8,Three & Four-Wheeler,PETROL/LPG,82
64838,2022,April,8,Heavy & Articulated Vehicle,DIESEL,247
64839,2022,April,8,Heavy & Articulated Vehicle,NOT APPLICABLE,60


In [69]:
df_vemision = pd.read_csv('dim_vahan.csv')

In [73]:
df_vemision

,Class,Vehicle,Fuel,CO₂ (g/km),CO (g/km),NOₓ (g/km),PM (g/km),HC (g/km),Notes/Methodology
0,Bus,BUS,DIESEL,708.27,1.06,7.64,0.010,0.10,"Real-world data for BS-VI Diesel Bus.[1, 2, 3]"
1,Bus,BUS,CNG ONLY,632.57,9.22,15.16,0.023,7.80,"Based on BS-III data for HCV CNG Bus.[4, 5]"
2,Bus,BUS,ELECTRIC(BOV),0.00,0.00,0.0,0.000,0.0,Zero tailpipe emissions.
3,Bus,BUS,PETROL,750.00,4.00,0.46,0.010,0.16,Estimated based on BS-VI HDV Positive Ignition...
4,Bus,BUS,FUEL CELL HYDROGEN,0.00,0.00,0.0,0.000,0.0,Zero tailpipe emissions.
...,...,...,...,...,...,...,...,...,...
328,Special & Emergency Vehicle,ADAPTED VEHICLE,PETROL/CNG,101.59,1.02,0.03,0.001,0.19,"Assumed same as base Petrol/CNG Car.[4, 5]"
329,Special & Emergency Vehicle,ADAPTED VEHICLE,ELECTRIC(BOV),0.00,0.00,0.0,0.000,0.0,Zero tailpipe emissions.
330,Special & Emergency Vehicle,ADAPTED VEHICLE,NOT APPLICABLE,0.00,0.00,0.0,0.000,0.0,Not an applicable fuel type.
331,Special & Emergency Vehicle,ADAPTED VEHICLE,DIESEL/HYBRID,111.89,0.40,0.06,0.004,0.14 (HC+NOx),Assumed same as base Diesel/Hybrid Car.


In [ ]:
df_vemision[df_vemision['HC (g/km)']=='3.76 (HC+NOx)']

,Class,Vehicle,Fuel,CO₂ (g/km),CO (g/km),NOₓ (g/km),PM (g/km),HC (g/km),Notes/Methodology
232,Heavy & Articulated Vehicle,TRACTOR-TROLLEY(COMMERCIAL),PETROL/ETHANOL,1000.0,4.0,4.7 (HC+NOx),0.025,3.76 (HC+NOx),"Assumed tractor is Agricultural type, using TR..."
268,Construction & Industrial Equipment,FORK LIFT,PETROL/ETHANOL,1320.0,4.0,4.7 (HC+NOx),0.025,3.76 (HC+NOx),Estimated ~20% CO/HC reduction from Petrol For...
270,Construction & Industrial Equipment,FORK LIFT,PETROL/HYBRID,1080.0,4.0,3.76 (HC+NOx),0.020,3.76 (HC+NOx),Estimated as 20% reduction from Petrol Fork Lift.
280,Agricultural Vehicle,AGRICULTURAL TRACTOR,PETROL/ETHANOL,1120.0,4.0,4.7 (HC+NOx),0.025,3.76 (HC+NOx),Estimated ~20% CO/HC reduction from Petrol Tra...


: 

In [80]:
df_vemision['HC (g/km)'].unique()

array(['0.10', '7.80', '0.0', '0.16', '7.0', '0.13', '0.08', '0.20',
       '0.17 (HC+NOx)', '0.19', '0.14 (HC+NOx)', '0.15', '0.05', '0.43',
       '0.04', '0.07', '0.35', '0.28', '0.40', '0.11', '0.09', '0.12',
       '0.25', '0.18 (HC+NOx)', '0.21', '0.64', '0.8', '1.0', '0.7',
       '4.7 (HC+NOx)', '3.76 (HC+NOx)', '0.3', '0.56'], dtype=object)

PROCESSED POPULATION_PROJECTION

In [58]:
df_projection = pd.read_csv('dataset/population_projection.csv')
df_projection

,year,month,state,gender,value,unit,note
0,2036,October,West Bengal,Total,43964,value in Thousands,NaN
1,2036,October,West Bengal,Male,22615,value in Thousands,NaN
2,2036,October,West Bengal,Female,21349,value in Thousands,NaN
3,2036,October,Uttarakhand,Total,5506,value in Thousands,NaN
4,2036,October,Uttarakhand,Male,2922,value in Thousands,NaN
...,...,...,...,...,...,...,...
8887,2011,July,Andaman and Nicobar Islands,Male,77,value in Thousands,NaN
8888,2011,July,Andaman and Nicobar Islands,Female,67,value in Thousands,NaN
8889,2011,July,All India,Total,380145,value in Thousands,NaN
8890,2011,July,All India,Male,197066,value in Thousands,NaN


In [59]:
df_projection.drop('unit',axis=1,inplace=True)
df_projection.drop('note',axis=1,inplace=True)

In [60]:
df_projection.loc[:,'state'] = df_projection['state'].replace('Daman and Diu','Dadra and Nagar Haveli and Daman and Diu')
df_projection.loc[:,'state'] = df_projection['state'].replace('Dadra and Nagar Haveli','Dadra and Nagar Haveli and Daman and Diu')
df_projection.loc[:,'state'] = df_projection['state'].replace(dict_state)
df_projection['state'].unique()

array([8, 12, 16, 28, 20, 13, 31, 15, 17, 21, 9, 27, 18, 22, 6, 1, 3, 36,
       35, 7, 10, 24, 32, 23, 25, 11, 33, 29, 34, 4, 19, 2, 5, 26, 14, 30,
       37], dtype=object)

In [61]:
df_projection.rename(columns={'state':'state_id','gender':'gender_id'},inplace=True)

In [62]:
dict_gender = {'Male':1,'Female':2,'Total':3}
#df_gender = pd.DataFrame(dict_gender.items(),columns=['gender','gender_id'])
#df_gender.to_csv('processed_dataset/dim_gender.csv',index=False,header=True)
df_projection.loc[:,'gender_id'] = df_projection['gender_id'].replace(dict_gender)

In [63]:
df_projection['pp_id'] = df_projection.reset_index().index + 1

In [64]:
#df_projection.to_csv('processed_dataset/fact_pop_pojection.csv',index=False,header=True)

Create dim_date

In [65]:
dim_date = pd.DataFrame({'date':pd.date_range(start='2022-01-01',end='2036-12-31')})

In [66]:
dim_date['year'] = dim_date['date'].dt.year
dim_date['month'] = dim_date['date'].dt.strftime('%B')
dim_date['day'] = dim_date['date'].dt.day
dim_date['week_of_year'] = dim_date['date'].dt.isocalendar().week
#dim_date.to_csv('processed_dataset/dim_date.csv',index=False,header=True)